In [1]:
# DataFrame sederhana dan operasi dasar
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('HandsOnPertemuan3').getOrCreate()

data = [('James', 'Sales', 3000),
        ('Michael', 'Sales', 4600),
        ('Robert', 'Sales', 4100),
        ('Maria', 'Finance', 3000),
        ('Yasa', 'CEO', 15000),
        ('Franklin', 'Finance', 4000),
        ('Thorfinn', 'Analyst', 5000)]

columns = ['EmployeeName', 'Department', 'Salary']

df = spark.createDataFrame(data, schema=columns)
df.show()

+------------+----------+------+
|EmployeeName|Department|Salary|
+------------+----------+------+
|       James|     Sales|  3000|
|     Michael|     Sales|  4600|
|      Robert|     Sales|  4100|
|       Maria|   Finance|  3000|
|        Yasa|       CEO| 15000|
|    Franklin|   Finance|  4000|
|    Thorfinn|   Analyst|  5000|
+------------+----------+------+



In [10]:
# Operasi transformasi DataFrame
df.select('EmployeeName', 'Salary').show()
df.filter(df['Salary'] > 3000).show()
df.groupBy('Department').avg('Salary').show()
df.groupBy("Department").agg(
    F.mean("Salary").alias("AvgSalary"),
    F.max("Salary").alias("MaxSalary"),
    F.min("Salary").alias("MinSalary"),
    F.sum("Salary").alias("TotalSalary"),
    F.count("Salary").alias("CountSalary")

).show()

+------------+------+
|EmployeeName|Salary|
+------------+------+
|       James|  3000|
|     Michael|  4600|
|      Robert|  4100|
|       Maria|  3000|
|        Yasa| 15000|
|    Franklin|  4000|
|    Thorfinn|  5000|
+------------+------+

+------------+----------+------+-----------+-----------------+
|EmployeeName|Department|Salary|SalaryBonus|TotalCompensation|
+------------+----------+------+-----------+-----------------+
|     Michael|     Sales|  4600|      460.0|           5060.0|
|      Robert|     Sales|  4100|      410.0|           4510.0|
|        Yasa|       CEO| 15000|     1500.0|          16500.0|
|    Franklin|   Finance|  4000|      400.0|           4400.0|
|    Thorfinn|   Analyst|  5000|      500.0|           5500.0|
+------------+----------+------+-----------+-----------------+

+----------+-----------+
|Department|avg(Salary)|
+----------+-----------+
|     Sales|     3900.0|
|   Finance|     3500.0|
|   Analyst|     5000.0|
|       CEO|    15000.0|
+----------+--

In [4]:
# Kolom SalaryBonus dihitung dari 10% gaji
df = df.withColumn("SalaryBonus", df["Salary"] * 0.1)

# Kolom TotalCompensation
df = df.withColumn("TotalCompensation", df["Salary"] + df["SalaryBonus"])
df.show()

+------------+----------+------+-----------+-----------------+
|EmployeeName|Department|Salary|SalaryBonus|TotalCompensation|
+------------+----------+------+-----------+-----------------+
|       James|     Sales|  3000|      300.0|           3300.0|
|     Michael|     Sales|  4600|      460.0|           5060.0|
|      Robert|     Sales|  4100|      410.0|           4510.0|
|       Maria|   Finance|  3000|      300.0|           3300.0|
|        Yasa|       CEO| 15000|     1500.0|          16500.0|
|    Franklin|   Finance|  4000|      400.0|           4400.0|
|    Thorfinn|   Analyst|  5000|      500.0|           5500.0|
+------------+----------+------+-----------+-----------------+



In [7]:
# Penggunaan window functions
from pyspark.sql.window import Window
from pyspark.sql import functions as F

windowSpec = Window.partitionBy('Department').orderBy('Salary')
df.withColumn('Rank', F.rank().over(windowSpec)).show()

+------------+----------+------+-----------+-----------------+----+
|EmployeeName|Department|Salary|SalaryBonus|TotalCompensation|Rank|
+------------+----------+------+-----------+-----------------+----+
|    Thorfinn|   Analyst|  5000|      500.0|           5500.0|   1|
|        Yasa|       CEO| 15000|     1500.0|          16500.0|   1|
|       Maria|   Finance|  3000|      300.0|           3300.0|   1|
|    Franklin|   Finance|  4000|      400.0|           4400.0|   2|
|       James|     Sales|  3000|      300.0|           3300.0|   1|
|      Robert|     Sales|  4100|      410.0|           4510.0|   2|
|     Michael|     Sales|  4600|      460.0|           5060.0|   3|
+------------+----------+------+-----------+-----------------+----+



In [21]:
from google.colab import files
import_file = files.upload()

Saving solar_data_khulna_from_jan_2014_to_nov_2022.csv to solar_data_khulna_from_jan_2014_to_nov_2022.csv


In [35]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, month, avg
from pyspark.sql.functions import concat_ws, to_date
spark = SparkSession.builder.appName('Analisis Tenaga Matahari').getOrCreate()


df = spark.read.csv("solar_data_khulna_from_jan_2014_to_nov_2022.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5)

df = df.withColumn("Date", to_date(concat_ws("-", col("Year"), col("Month"), col("Day")), "yyyy-M-d"))
df.select("Year", "Month", "Day", "Date", "Irradiance").show(5)

df = df.withColumn("Date", col("Date").cast("date"))

df = df.withColumn("Year", year(col("Date")))\
       .withColumn("Month", month(col("Date")))

df.describe("Irradiance").show()

yearly = df.groupBy("Year").agg(avg("Irradiance").alias("AvgIrradiance"))
yearly.orderBy("Year").show()

montly = df.groupBy("Month").agg(avg("Irradiance").alias("AvgIrradiance"))
montly.orderBy("Month").show()

root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Hour: integer (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Irradiance: double (nullable = true)

+----+-----+---+----+-----------+----------+
|Year|Month|Day|Hour|Temperature|Irradiance|
+----+-----+---+----+-----------+----------+
|2014|    1|  1|   6|       9.44|       0.0|
|2014|    1|  1|   7|      11.87|     99.09|
|2014|    1|  1|   8|      14.55|    290.91|
|2014|    1|  1|   9|      17.81|    492.71|
|2014|    1|  1|  10|      21.96|    647.74|
+----+-----+---+----+-----------+----------+
only showing top 5 rows

+----+-----+---+----------+----------+
|Year|Month|Day|      Date|Irradiance|
+----+-----+---+----------+----------+
|2014|    1|  1|2014-01-01|       0.0|
|2014|    1|  1|2014-01-01|     99.09|
|2014|    1|  1|2014-01-01|    290.91|
|2014|    1|  1|2014-01-01|    492.71|
|2014|    1|  1|2014-01-01|    647.74|
+----+-----+---+---